In [1]:
#@title Install DEAP
!pip install deap --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.0/136.0 kB 2.6 MB/s eta 0:00:00


In [2]:
import random
import numpy as np
from deap import base, creator, tools, algorithms

# Define the fitness class (single objective maximization)
creator.create("FitnessMin", base.Fitness, weights=(-1.0,)) # if 1.0, maxmization
creator.create("Individual", list, fitness=creator.FitnessMin)

# Define a sphere function to be minimized
def sphere(x):
    return x[0]**2 + x[1]**2,

def polynomial(x):  # minima at y=0
    return (x[0] ** 3 - 2 * x[0] ** 2 - x[0] + 2) ** 2,

def rosenbrock(x):
    return (1-x[0])**2 + 100*(x[1]-x[0]**2)**2,

def booth(x): # minima at (1,3)
    return ((x[0] + 2 * x[1] - 7) ** 2 + (2 * x[0] + x[1] - 5) ** 2, )

def ackley(x): # minima at (0,0)
    a = 20
    b = 0.2
    c = 2 * np.pi
    d = 2
    sum1 = -a * np.exp(-b * np.sqrt((1 / d) * (x[0] ** 2 + x[1] ** 2)))
    sum2 = -np.exp((1 / d) * (np.cos(c * x[0]) + np.cos(c * x[1])))
    return (sum1 + sum2 + a + np.exp(1),) # return as a tuple

In [ ]:
def deap_run(eval_func, numVar):
  # Create the toolbox
  toolbox = base.Toolbox()
  toolbox.register("attr_float", random.uniform, -5, 5)
  toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_float, n=numVar)
  toolbox.register("population", tools.initRepeat, list, toolbox.individual)
  toolbox.register("evaluate", eval_func)
  if numVar > 1:
    toolbox.register("mate", tools.cxTwoPoint)  # Use cxTwoPoint for numVar > 1
    toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)
  else:
    toolbox.register("mate", tools.cxUniform, indpb=0.5) # added this for single gene
    #toolbox.register("mate", tools.cxOnePoint) # this also works
    # Use a suitable operator for numVar = 1, e.g., mutGaussian
    toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=1, indpb=0.1)
  toolbox.register("select", tools.selTournament, tournsize=3)

  # Run the algorithm
  successes = 0
  trials = 3
  for tr in range(trials):
    population = toolbox.population(n=300)
    for gen in range(500):
      # varAnd: apply both variation operators—crossover and mutation—to a population of individuals
      offspring = algorithms.varAnd(population, toolbox, cxpb=0.5, mutpb=0.1)
      fitnesses = map(toolbox.evaluate, offspring) # see below for map()
      for ind, fit in zip(offspring, fitnesses):
        ind.fitness.values = fit
      population = toolbox.select(offspring, k=len(population))

    # Get the best individual
    best_ind = tools.selBest(population, k=1)[0]
    print(f"---------- Trial: {tr} ")
    print("Best individual:", best_ind)
    if best_ind.fitness.values[0] < 0.00005:
      successes += 1
    print("Best fitness:", best_ind.fitness.values[0])
  print(f"\nSystem Success %: {successes*100/trials}")

########################### M A I N ###########################
print("\n***************** Sphere Function *****************")
deap_run(sphere, 2)

print("\n***************** Polynomial Function *****************")
deap_run(polynomial, 1)

print("\n***************** Rosenbrock Function *****************")
deap_run(rosenbrock, 2)

print("\n***************** Booth Function *****************")
deap_run(booth, 2)

print("\n***************** Ackley Function *****************")
deap_run(ackley, 2)



***************** Sphere Function *****************
---------- Trial: 0 
Best individual: [0.0, 0.0]
Best fitness: 0.0
---------- Trial: 1 
Best individual: [0.0, 0.0]
Best fitness: 0.0
---------- Trial: 2 
Best individual: [0.0, 0.0]
Best fitness: 0.0

System Success %: 100.0

***************** Polynomial Function *****************
---------- Trial: 0 
Best individual: [-1.000660481863764]
Best fitness: 1.5721801137011518e-05
---------- Trial: 1 
Best individual: [1.999264165587377]
Best fitness: 4.863514931891199e-06
---------- Trial: 2 
Best individual: [0.9988675278589487]
Best fitness: 5.135777213995181e-06

System Success %: 100.0

***************** Rosenbrock Function *****************
---------- Trial: 0 
Best individual: [1.4414566287897674, 2.112232064682278]
Best fitness: 0.31345985833079726
---------- Trial: 1 
Best individual: [1.2327010010444521, 1.518654450795153]
Best fitness: 0.054230271904769055
---------- Trial: 2 
Best individual: [1.1499663311228296, 1.30365013562

## Python Review

In [ ]:
def no_trailing_comma(x):
  return x + [10]

def trailing_comma(x):
  return x + [10],

data = [1, 2, 3]

no_comma_value = no_trailing_comma(data)
comma_value = trailing_comma(data)

print(f"{no_comma_value}  The return type is {type(no_comma_value)}")
print(f"{comma_value}  The return type is {type(comma_value)}")

[1, 2, 3, 10]  The return type is <class 'list'>
([1, 2, 3, 10],)  The return type is <class 'tuple'>


In [ ]:
# zip() combines multiple iterables such as lists, tuples, strings, dict etc,
# into a single iterator of tuples
names = ['John', 'Alice', 'Bob', 'Lucy']
scores = [85, 90, 78, 92]

res = zip(names, scores) # res is an object
print(list(res))

[('John', 85), ('Alice', 90), ('Bob', 78), ('Lucy', 92)]


In [ ]:
# The map() function in Python applies a given function to all items in an iterable
# (like a list or tuple) and returns an iterator with the results.
def square(x):
    return x * x

# List of numbers
numbers = [1, 2, 3, 4, 5]

# Use map to apply 'square' to each element in 'numbers'
squared_numbers = map(square, numbers)

# Convert the map object to a list to see the results
print(list(squared_numbers))  # Output: [1, 4, 9, 16, 25]

[1, 4, 9, 16, 25]


In [ ]:
# using lamda function
numbers = [1, 2, 3, 4, 5]
squared_numbers = map(lambda x: x * x, numbers)
print(list(squared_numbers))  # Output: [1, 4, 9, 16, 25]

[1, 4, 9, 16, 25]
